# Day 15: Model Parallelism — Tensor & Expert
> *Inference Engineering* — Chapter 5.4 | Philip Kiely, Baseten Books 2026

**Layer:** Runtime | **Prerequisite:** Day 03 (MoE), Day 10 (Disaggregation)


## Concept Overview

Model parallelism splits a model across multiple GPUs when it doesn't fit on one device. Tensor parallelism (TP) shards individual weight matrices across GPUs — each GPU holds 1/N of each layer's weights and computes a partial result, requiring an AllReduce after each layer. Expert parallelism (EP) in MoE models places different experts on different GPUs — a token is routed to the GPU holding its expert via All-to-All communication. These two strategies can be combined for the largest models.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU count: {torch.cuda.device_count()}')


## 1. Tensor Parallelism — Column and Row Splitting

For a linear layer $Y = XW + b$:
- **Column parallel:** Split $W$ by columns. Each GPU computes $Y_i = X W_i$. Results are concatenated: $Y = [Y_1, Y_2, \ldots]$.
- **Row parallel:** Split $W$ by rows, split $X$ by columns. Each GPU computes $Y_i = X_i W_i$. Results are summed: $Y = \sum_i Y_i$ (AllReduce).

Megatron-LM pairs column-parallel with row-parallel to minimize communication: one AllReduce per transformer layer instead of one per matmul.


In [ ]:
class ColumnParallelLinear(nn.Module):
    """Simulate column-parallel linear layer (no actual multi-GPU)."""
    def __init__(self, in_features, out_features, tp_size):
        super().__init__()
        assert out_features % tp_size == 0
        self.tp_size = tp_size
        self.out_per_rank = out_features // tp_size
        # Each rank holds out_per_rank columns
        self.weights = nn.ParameterList([
            nn.Parameter(torch.randn(self.out_per_rank, in_features) * 0.01)
            for _ in range(tp_size)
        ])

    def forward(self, x):
        # Simulate each rank's partial computation
        partial_outputs = [F.linear(x, w) for w in self.weights]
        return torch.cat(partial_outputs, dim=-1)  # all-gather equivalent

class RowParallelLinear(nn.Module):
    """Simulate row-parallel linear layer."""
    def __init__(self, in_features, out_features, tp_size):
        super().__init__()
        assert in_features % tp_size == 0
        self.tp_size = tp_size
        self.in_per_rank = in_features // tp_size
        self.weights = nn.ParameterList([
            nn.Parameter(torch.randn(out_features, self.in_per_rank) * 0.01)
            for _ in range(tp_size)
        ])

    def forward(self, x):
        # Scatter input across ranks
        x_parts = x.chunk(self.tp_size, dim=-1)
        partial_outputs = [F.linear(xp, w) for xp, w in zip(x_parts, self.weights)]
        return sum(partial_outputs)  # all-reduce equivalent

tp_size = 4
col = ColumnParallelLinear(512, 2048, tp_size)
row = RowParallelLinear(2048, 512, tp_size)
x = torch.randn(8, 32, 512)

out_col = col(x)
out_row = row(out_col)
print(f'Input:              {x.shape}')
print(f'Column parallel:    {out_col.shape} (each rank: {2048//tp_size} features)')
print(f'Row parallel:       {out_row.shape}')
print(f'Communication: 1 AllReduce after row-parallel (vs {tp_size} without Megatron pairing)')

# Memory savings
total_params = 512*2048 + 2048*512
params_per_rank = total_params // tp_size
print(f'\nParams per rank: {params_per_rank:,} ({100/tp_size:.0f}% of total {total_params:,})')


## 2. Communication Cost Analysis

AllReduce cost determines TP scalability. For ring AllReduce on N GPUs with D bytes:
$t_{\text{AllReduce}} \approx 2 \cdot \frac{N-1}{N} \cdot \frac{D}{\text{bandwidth}}$


In [ ]:
def allreduce_time_ms(tensor_bytes, tp_size, bandwidth_gb_s):
    """Ring AllReduce time estimate."""
    return 2 * (tp_size - 1) / tp_size * tensor_bytes / (bandwidth_gb_s * 1e9) * 1000

# For a Llama-3-70B forward pass
d_model = 8192
batch_seq_tokens = 512  # batch=4, seq=128
tensor_bytes = batch_seq_tokens * d_model * 2  # FP16

print('AllReduce overhead per transformer layer (Llama-3-70B, 512 tokens):')
print(f'{"TP Size":>8} {"NVLink (900GB/s)":>18} {"InfiniBand (25GB/s)":>22} {"PCIe (32GB/s)":>16}')
for tp in [1, 2, 4, 8]:
    t_nvlink = allreduce_time_ms(tensor_bytes, tp, 900) if tp > 1 else 0
    t_ib = allreduce_time_ms(tensor_bytes, tp, 25) if tp > 1 else 0
    t_pcie = allreduce_time_ms(tensor_bytes, tp, 32) if tp > 1 else 0
    print(f'{tp:>8} {t_nvlink:>18.3f} {t_ib:>22.3f} {t_pcie:>16.3f}')

print('\nTP is efficient on NVLink (intra-node) but expensive on InfiniBand (inter-node)')


## 3. Expert Parallelism

In MoE models, each GPU hosts a subset of experts. The All-to-All communication pattern: tokens are routed to remote GPUs holding their target experts, computed there, and sent back.


In [ ]:
def simulate_expert_parallel(num_experts, ep_size, num_tokens, top_k=2):
    """Simulate expert parallelism routing."""
    experts_per_rank = num_experts // ep_size
    np.random.seed(42)

    # Route tokens: each gets top-k experts
    assignments = np.random.randint(0, num_experts, size=(num_tokens, top_k))
    rank_assignments = assignments // experts_per_rank

    # Count cross-rank traffic
    local_tokens = 0
    remote_tokens = 0
    for token_idx in range(num_tokens):
        home_rank = token_idx % ep_size  # simplified
        for expert_rank in rank_assignments[token_idx]:
            if expert_rank == home_rank:
                local_tokens += 1
            else:
                remote_tokens += 1

    a2a_bytes = remote_tokens * 4096 * 2  # tokens × hidden_dim × FP16
    return local_tokens, remote_tokens, a2a_bytes

print('Expert Parallelism Communication Analysis (Mixtral-8x7B config):')
print(f'{"EP Size":>8} {"Local Tokens":>14} {"Remote Tokens":>15} {"A2A Data (MB)":>15}')
for ep_size in [1, 2, 4, 8]:
    local, remote, a2a = simulate_expert_parallel(num_experts=8, ep_size=ep_size, num_tokens=512)
    print(f'{ep_size:>8} {local:>14} {remote:>15} {a2a/1e6:>15.1f}')


## Experiments: Try These

1. **TP scaling efficiency**: Measure actual matmul time across TP sizes 1, 2, 4 on your hardware. At what TP size does communication overhead dominate compute savings?
2. **Pipeline parallelism**: Implement a simple pipeline parallel transformer (2 GPUs, each holds half the layers). Measure pipeline bubble overhead.
3. **TP + EP combined**: For a 64-expert MoE model on 8 GPUs with EP=8 and TP=2, draw the communication pattern. How many cross-GPU messages per forward pass?


## Key Takeaways

- Tensor parallelism splits individual weight matrices across GPUs, requiring one AllReduce per transformer layer — efficient on NVLink, expensive on InfiniBand.
- Megatron-style TP pairs column-parallel and row-parallel to reduce AllReduce count from 2 per matmul pair to 1 per transformer block.
- Expert parallelism places different experts on different GPUs; All-to-All communication routes tokens to the correct expert GPU.
- TP is best within a node (NVLink); pipeline parallelism is better across nodes (no AllReduce, just point-to-point activations).

## References
- *Inference Engineering* Ch 5.4 — Philip Kiely, Baseten Books 2026
- Shoeybi et al. (2019), "Megatron-LM: Training Multi-Billion Parameter Language Models"
- Lepikhin et al. (2020), "GShard: Scaling Giant Models with Conditional Computation"
